# Problema X21B11T1 — Solução via Funções de Green

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x21b11t1.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição do problema

Placa plana de espessura $L$, com:

- **Contorno $x = 0$** (tipo 2): fluxo de calor prescrito $q''_0$
- **Contorno $x = L$** (tipo 1): temperatura prescrita $T_L$
- **Condição inicial:** $T(x,0) = T_i$

A notação X21B11T1 indica: geometria plana (X), fluxo prescrito em $x=0$ (tipo 2), temperatura prescrita em $x=L$ (tipo 1), ambas as condições de contorno não nulas (B11) e condição inicial não nula (TL).

## Solução analítica

Introduzindo $\Theta(x,t) = T(x,t) - T_L$ para homogeneizar a condição em $x=L$, a solução via funções de Green contém **duas contribuições independentes**:

$$T(x,t) = T_L + \underbrace{\frac{q''_0}{k}(L - x)}_{\text{regime permanente}}
\underbrace{- \frac{2\alpha}{L}\frac{q''_0}{k}\sum_{m=1}^{\infty} \frac{e^{-A_m t}\cos(\beta_m x/L)}{A_m}}_{\text{transiente do fluxo}}
\underbrace{+ 2(T_i-T_L)\sum_{m=1}^{\infty}\frac{\sin(\beta_m)}{\beta_m}\,e^{-A_m t}\cos(\beta_m x/L)}_{\text{transiente da CI}}$$

com:
$$\beta_m = \pi\!\left(m - \tfrac{1}{2}\right), \qquad A_m = \frac{\alpha\,\beta_m^2}{L^2}, \qquad \sin(\beta_m) = (-1)^{m+1}, \qquad m = 1, 2, 3, \ldots$$

**Interpretação física dos três termos:**
- **Regime permanente** $T_L + q''_0(L-x)/k$: perfil linear atingido para $t \to \infty$, independente de $T_i$.
- **Transiente do fluxo**: decai exponencialmente e reflete a resposta ao fluxo $q''_0$ aplicado em $x=0$.
- **Transiente da CI**: decai exponencialmente e reflete o afastamento entre a temperatura inicial $T_i$ e a temperatura de referência $T_L$. Se $T_i = T_L$, este termo é nulo.

A série infinita é truncada em $M$ termos suficientemente grande para garantir convergência.

## Bibliotecas utilizadas

Para avaliar a solução analítica computacionalmente, utilizamos duas bibliotecas Python:

- **NumPy** (`numpy`): biblioteca para computação numérica. Fornece vetores e matrizes eficientes e funções matemáticas como `np.sqrt`, `np.exp`, `np.cos`, `np.linspace`.
- **Matplotlib** (`matplotlib.pyplot`): biblioteca para criação de gráficos 2D.

A linha `import numpy as np` importa a biblioteca e cria o atalho `np` — assim escrevemos `np.cos` em vez de `numpy.cos`.

In [ ]:
import numpy as np              # computação numérica
import matplotlib.pyplot as plt # gráficos

## Parâmetros do problema

Os parâmetros físicos são armazenados como **variáveis** Python. Uma variável é criada com o sinal `=`, por exemplo `L = 66e-4` define $L = 0{,}0066$ m (a notação `66e-4` é equivalente a $66 \times 10^{-4}$). Os comentários após `#` não são executados e servem apenas para documentar o código.

Note que `Ti` e `TL` são parâmetros **independentes**: `TL` é a temperatura prescrita na superfície $x=L$ e `Ti` é a temperatura inicial da placa. Quando `Ti == TL`, o termo de condição inicial se anula e o problema se reduz ao caso particular X21B10T0.

In [ ]:
L     = 66e-4        # espessura da placa [m]  (66e-4 = 0,0066 m)
alpha = 1.93e-7      # difusividade térmica [m²/s]
k     = 0.81         # condutividade térmica [W/(m·K)]
q0    = 1e3          # fluxo de calor imposto em x=0 [W/m²]  (1e3 = 1000)
TL    = 40.0         # temperatura prescrita em x=L [°C]
Ti    = 20.0         # temperatura inicial uniforme da placa [°C]

M     = 200          # número de termos da série (truncamento)

## Autovalores, vetores de tempo e posição

- `np.arange(1, M+1)` cria um vetor de inteiros de 1 até M: `[1, 2, 3, ..., M]`. É semelhante ao `range` do Python, mas retorna um vetor NumPy sobre o qual operações matemáticas podem ser aplicadas diretamente.
- `np.linspace(inicio, fim, N)` cria um vetor com `N` valores reais igualmente espaçados entre `inicio` e `fim`.
- As listas `x_vals` e `x_names` armazenam, respectivamente, as posições numéricas e os rótulos para a legenda.

In [ ]:
m_arr = np.arange(1, M + 1)          # índices m = 1, 2, ..., M
beta  = np.pi * (m_arr - 0.5)        # β_m = π(m − 1/2)  (vetor de M autovalores)
A     = alpha * beta**2 / L**2       # A_m = α β_m² / L²  (vetor de M coeficientes)
sin_b = np.sin(beta)                 # sin(β_m) = (-1)^(m+1), alterna +1, -1, +1, ...

t_max = 500                          # tempo máximo [s]
t     = np.linspace(0, t_max, 600)   # 600 instantes de tempo entre 0 e t_max

x_vals  = [0, L/4, L/2, 3*L/4, L]
x_names = ['$x = 0$', '$x = L/4$', '$x = L/2$', '$x = 3L/4$', '$x = L$']

## Definição da função de temperatura

Em Python, uma **função** é definida com a palavra-chave `def`, seguida do nome, dos parâmetros entre parênteses e de dois pontos. O corpo da função deve ser **indentado** (recuado com espaços ou Tab) — a indentação indica ao Python quais linhas pertencem à função. A palavra `return` especifica o valor devolvido.

```python
def nome_da_funcao(parametro):
    resultado = ...     # estas linhas estão dentro da função (indentadas)
    return resultado    # devolve o valor calculado
```

O cálculo da série é vetorizado usando `np.outer` e `np.sum`:
- `np.outer(A, t_vec)` cria uma matriz $M \times N_t$ onde o elemento $(i,j)$ é $A_i \cdot t_j$.
- `np.sum(..., axis=0)` soma ao longo dos $M$ termos da série, resultando em um vetor de tamanho $N_t$.

A função `temperatura` calcula os três termos da solução separadamente para evidenciar a contribuição de cada um:

In [ ]:
def temperatura(x, t_vec):
    """Retorna T(x, t) para um escalar x e um vetor de tempos t_vec."""
    cos_m   = np.cos(beta * x / L)           # cos(β_m x/L), vetor de M valores
    exp_mat = np.exp(-np.outer(A, t_vec))     # e^{-A_m t}, matriz M × N_t

    # Regime permanente: perfil linear de temperatura
    permanente = (q0/k) * (L - x)

    # Transiente do fluxo: resposta ao fluxo q0 aplicado em x=0
    trans_fluxo = (2*alpha/L) * (q0/k) * np.sum(
        (cos_m / A)[:, None] * exp_mat,
        axis=0                                # soma os M termos da série
    )

    # Transiente da CI: resposta ao afastamento entre Ti e TL
    # sin(β_m) = (-1)^(m+1) para os autovalores do problema X21
    trans_ci = 2 * (Ti - TL) * np.sum(
        (sin_b * cos_m / beta)[:, None] * exp_mat,
        axis=0
    )

    return TL + permanente - trans_fluxo + trans_ci

## Gráfico do campo de temperatura

O laço `for` usa `zip(x_vals, x_names)` para percorrer as duas listas ao mesmo tempo.
`zip` funciona como um zíper: emparelha o primeiro elemento de cada lista, depois o segundo, e assim por diante.
Assim, a cada passo do laço, `x` assume um valor de posição e `nome` assume o rótulo correspondente:
no 1.º passo `x = 0` e `nome = '$x = 0$'`; no 2.º, `x = L/4` e `nome = '$x = L/4$'`; e assim até o 5.º.

A função `ax.plot` recebe os vetores $t$ e $T(x,t)$ e o parâmetro `label` (o texto que aparece na legenda).

In [ ]:
fig, ax = plt.subplots()

for x, nome in zip(x_vals, x_names):
    ax.plot(t, temperatura(x, t), label=nome)

ax.set_xlabel('Tempo (s)')
ax.set_ylabel('$T(x, t)$  (°C)')
ax.set_title(f'Problema X21B11T1 — $T_i = {Ti}\\,°C$, $T_L = {TL}\\,°C$')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Verificação: caso particular $T_i = T_L$

Quando `Ti == TL`, o termo de condição inicial se anula e o resultado deve coincidir com a solução X21B10T0 (apenas fluxo, sem contribuição da CI). O gráfico abaixo confirma esse comportamento ao forçar `Ti = TL`.

In [ ]:
Ti_orig = Ti           # guarda o valor atual de Ti
Ti = TL                # força Ti = TL para comparar com X21B10T0

fig, ax = plt.subplots()
for x, nome in zip(x_vals, x_names):
    ax.plot(t, temperatura(x, t), label=nome)

ax.set_xlabel('Tempo (s)')
ax.set_ylabel('$T(x, t)$  (°C)')
ax.set_title(f'Caso particular $T_i = T_L = {TL}\\,°C$ (equivale a X21B10T0)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

Ti = Ti_orig           # restaura o valor original de Ti